# Retail Sales Data Analysis — Python

## End-to-End Exploratory Data Analysis

**Objective:** Transform retail transaction data into clean, decision-ready business insights.

### Workflow
1. Load data
2. Inspect data quality
3. Clean and prepare data
4. Engineer analytical features
5. Calculate business KPIs
6. Analyze category, region, customer and shipping performance
7. Study trends and discount impact
8. Generate 12+ visualizations
9. Export analysis outputs
10. Produce actionable business recommendations


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("data/retail_sales.csv")
VISUAL_DIR = Path("visuals")
OUTPUT_DIR = Path("outputs")

VISUAL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Libraries loaded successfully.")


In [ ]:
# Load the real dataset if available.
# If it is not available, create a clearly-labelled demo dataset.

def create_demo_dataset(n=4500, seed=42):
    rng = np.random.default_rng(seed)

    categories = ["Technology", "Furniture", "Office Supplies"]
    subcats = {
        "Technology": ["Phones", "Computers", "Accessories", "Machines"],
        "Furniture": ["Chairs", "Tables", "Bookcases", "Furnishings"],
        "Office Supplies": ["Paper", "Binders", "Storage", "Art", "Labels"]
    }
    regions = ["West", "East", "Central", "South"]
    segments = ["Consumer", "Corporate", "Home Office"]
    ships = ["Standard Class", "Second Class", "First Class", "Same Day"]
    states = ["California", "New York", "Texas", "Florida", "Illinois",
              "Ohio", "Arizona", "Michigan", "Washington", "Georgia"]

    dates = pd.date_range("2023-01-01", "2025-12-31", freq="D")
    order_dates = rng.choice(dates, n)

    rows = []
    for i, d in enumerate(order_dates, 1):
        cat = rng.choice(categories, p=[0.32, 0.28, 0.40])
        sub = rng.choice(subcats[cat])
        qty = int(rng.integers(1, 8))
        price = float(rng.uniform(8, 650))
        discount = float(rng.choice([0, .05, .10, .15, .20, .30, .40, .50],
                                     p=[.10,.10,.20,.20,.20,.10,.07,.03]))
        sales = qty * price * (1 - discount)
        base_margin = {"Technology": .22, "Furniture": .10, "Office Supplies": .18}[cat]
        profit = sales * (base_margin - discount * .45 + rng.normal(0, .035))
        ship_mode = rng.choice(ships, p=[.60,.22,.14,.04])
        ship_days = {"Standard Class": 5, "Second Class": 3,
                     "First Class": 2, "Same Day": 0}[ship_mode]
        ship_date = d + pd.Timedelta(days=int(max(0, ship_days + rng.normal(0, 1))))
        rows.append([
            f"ORD-{i:05d}", d, ship_date, ship_mode,
            rng.choice(segments, p=[.55,.28,.17]),
            rng.choice(regions), rng.choice(states),
            cat, sub, qty, round(sales,2), discount, round(profit,2)
        ])

    return pd.DataFrame(rows, columns=[
        "order_id","order_date","ship_date","ship_mode","segment",
        "region","state","category","sub_category","quantity",
        "sales","discount","profit"
    ])

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded real dataset: {DATA_PATH}")
else:
    df = create_demo_dataset()
    print("No data/retail_sales.csv found.")
    print("A DEMO dataset was generated for testing. Replace it with your real dataset before publishing final portfolio results.")

print(f"Shape: {df.shape}")
display(df.head())


## 1. Data Quality Audit

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_%": (df.isna().mean()*100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_values", ascending=False)

display(quality)

print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# Standardize column names
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_", regex=False)
)

# Convert date fields
for col in ["order_date", "ship_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Remove exact duplicates
before = len(df)
df = df.drop_duplicates().copy()
removed = before - len(df)

# Fill common categorical missing values
for col in ["ship_mode", "segment", "region", "state", "category", "sub_category"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# Numeric conversion
for col in ["quantity", "sales", "discount", "profit"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Duplicate rows removed: {removed}")
print("Cleaned shape:", df.shape)
display(df.head())


## 2. Feature Engineering

In [ ]:
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["shipping_days"] = df["shipping_days"].clip(lower=0)

df["profit_margin"] = np.where(
    df["sales"] != 0,
    (df["profit"] / df["sales"]) * 100,
    0
)

df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.strftime("%b")
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)
df["quarter"] = df["order_date"].dt.to_period("Q").astype(str)

df["discount_band"] = pd.cut(
    df["discount"],
    bins=[-0.001, 0.10, 0.20, 0.30, 0.40, 1.00],
    labels=["0-10%", "10-20%", "20-30%", "30-40%", "40%+"]
)

print("Feature engineering completed.")
display(df[[
    "order_id","sales","profit","discount","profit_margin",
    "shipping_days","year","quarter","discount_band"
]].head())


## 3. Executive KPI Summary

In [ ]:
total_sales = df["sales"].sum()
total_profit = df["profit"].sum()
total_orders = df["order_id"].nunique()
total_units = df["quantity"].sum()
avg_order_value = total_sales / total_orders if total_orders else 0
overall_margin = (total_profit / total_sales * 100) if total_sales else 0
avg_discount = df["discount"].mean() * 100
avg_shipping = df["shipping_days"].mean()

kpis = pd.DataFrame({
    "KPI": [
        "Total Sales","Total Profit","Unique Orders","Units Sold",
        "Average Order Value","Profit Margin","Average Discount","Avg Shipping Days"
    ],
    "Value": [
        total_sales,total_profit,total_orders,total_units,
        avg_order_value,overall_margin,avg_discount,avg_shipping
    ]
})

display(kpis)

print(f"Total Sales: ₹{total_sales:,.2f}")
print(f"Total Profit: ₹{total_profit:,.2f}")
print(f"Unique Orders: {total_orders:,}")
print(f"Units Sold: {total_units:,}")
print(f"Average Order Value: ₹{avg_order_value:,.2f}")
print(f"Overall Profit Margin: {overall_margin:.2f}%")


## 4. Category Performance

In [ ]:
category_summary = (
    df.groupby("category")
      .agg(
          sales=("sales","sum"),
          profit=("profit","sum"),
          orders=("order_id","nunique"),
          quantity=("quantity","sum"),
          avg_discount=("discount","mean")
      )
      .assign(
          profit_margin=lambda x: x["profit"]/x["sales"]*100
      )
      .sort_values("sales", ascending=False)
)

display(category_summary.round(2))
category_summary.to_csv(OUTPUT_DIR/"category_summary.csv")

plt.figure(figsize=(10,6))
sns.barplot(data=category_summary.reset_index(), x="category", y="sales")
plt.title("Sales by Product Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(VISUAL_DIR/"01_sales_profit_by_category.png", dpi=150)
plt.show()


## 5. Regional Performance

In [ ]:
region_summary = (
    df.groupby("region")
      .agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique"))
      .sort_values("sales", ascending=False)
)

display(region_summary.round(2))
region_summary.to_csv(OUTPUT_DIR/"region_summary.csv")

plt.figure(figsize=(9,6))
sns.barplot(data=region_summary.reset_index(), x="region", y="sales")
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Sales")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"02_sales_by_region.png", dpi=150)
plt.show()


## 6. Monthly Sales & Profit Trends

In [ ]:
monthly = (
    df.groupby("year_month")
      .agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique"))
      .reset_index()
)

monthly.to_csv(OUTPUT_DIR/"monthly_sales_summary.csv", index=False)

plt.figure(figsize=(14,6))
plt.plot(monthly["year_month"], monthly["sales"], marker="o", label="Sales")
plt.xticks(rotation=60)
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.savefig(VISUAL_DIR/"03_monthly_sales_trend.png", dpi=150)
plt.show()

plt.figure(figsize=(14,6))
plt.plot(monthly["year_month"], monthly["profit"], marker="o", label="Profit")
plt.xticks(rotation=60)
plt.title("Monthly Profit Trend")
plt.xlabel("Month")
plt.ylabel("Profit")
plt.legend()
plt.tight_layout()
plt.savefig(VISUAL_DIR/"04_monthly_profit_trend.png", dpi=150)
plt.show()

display(monthly.tail(12).round(2))


## 7. Sub-Category Profitability

In [ ]:
subcategory = (
    df.groupby("sub_category")
      .agg(sales=("sales","sum"), profit=("profit","sum"))
      .sort_values("profit", ascending=False)
)

display(subcategory.round(2))

plt.figure(figsize=(12,7))
sns.barplot(data=subcategory.reset_index(), x="profit", y="sub_category")
plt.title("Profit by Sub-Category")
plt.xlabel("Profit")
plt.ylabel("Sub-Category")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"06_profit_by_subcategory.png", dpi=150)
plt.show()


## 8. Discount vs Profitability

In [ ]:
discount_summary = (
    df.groupby("discount_band", observed=False)
      .agg(
          sales=("sales","sum"),
          profit=("profit","sum"),
          orders=("order_id","nunique"),
          avg_margin=("profit_margin","mean")
      )
      .reset_index()
)

display(discount_summary.round(2))

plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df.sample(min(len(df), 2500), random_state=42),
    x="discount", y="profit_margin", alpha=0.5
)
plt.axhline(0, linestyle="--")
plt.title("Discount vs Profit Margin")
plt.xlabel("Discount")
plt.ylabel("Profit Margin (%)")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"05_discount_vs_profit_margin.png", dpi=150)
plt.show()


## 9. Customer Segment Analysis

In [ ]:
segment = (
    df.groupby("segment")
      .agg(sales=("sales","sum"), profit=("profit","sum"), orders=("order_id","nunique"))
      .sort_values("sales", ascending=False)
)

segment["sales_share_%"] = segment["sales"] / segment["sales"].sum() * 100
display(segment.round(2))

plt.figure(figsize=(8,6))
plt.pie(segment["sales"], labels=segment.index, autopct="%1.1f%%")
plt.title("Sales Share by Customer Segment")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"07_sales_share_by_segment.png", dpi=150)
plt.show()


## 10. Shipping Performance

In [ ]:
shipping = (
    df.groupby("ship_mode")
      .agg(
          orders=("order_id","nunique"),
          avg_shipping_days=("shipping_days","mean"),
          sales=("sales","sum"),
          profit=("profit","sum")
      )
      .sort_values("orders", ascending=False)
)

display(shipping.round(2))

plt.figure(figsize=(10,6))
sns.barplot(data=shipping.reset_index(), x="ship_mode", y="avg_shipping_days")
plt.title("Average Shipping Days by Shipping Mode")
plt.xlabel("Shipping Mode")
plt.ylabel("Average Shipping Days")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(VISUAL_DIR/"08_shipping_days_by_mode.png", dpi=150)
plt.show()


## 11. Top States by Sales

In [ ]:
top_states = (
    df.groupby("state")
      .agg(sales=("sales","sum"), profit=("profit","sum"))
      .sort_values("sales", ascending=False)
      .head(10)
)

display(top_states.round(2))

plt.figure(figsize=(11,7))
sns.barplot(data=top_states.reset_index(), x="sales", y="state")
plt.title("Top 10 States by Sales")
plt.xlabel("Sales")
plt.ylabel("State")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"09_top_states_by_sales.png", dpi=150)
plt.show()


## 12. Profit Margin by Category

In [ ]:
margin_category = (
    df.groupby("category")
      .agg(sales=("sales","sum"), profit=("profit","sum"))
)

margin_category["profit_margin_%"] = margin_category["profit"] / margin_category["sales"] * 100

display(margin_category.round(2))

plt.figure(figsize=(9,6))
sns.barplot(data=margin_category.reset_index(), x="category", y="profit_margin_%")
plt.title("Profit Margin by Category")
plt.xlabel("Category")
plt.ylabel("Profit Margin (%)")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"10_profit_margin_by_category.png", dpi=150)
plt.show()


## 13. Quantity vs Sales

In [ ]:
sample = df.sample(min(len(df), 2500), random_state=42)

plt.figure(figsize=(9,6))
sns.scatterplot(data=sample, x="quantity", y="sales", alpha=0.5)
plt.title("Quantity vs Sales")
plt.xlabel("Quantity")
plt.ylabel("Sales")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"11_quantity_vs_sales.png", dpi=150)
plt.show()

print("Correlation between quantity and sales:",
      round(df["quantity"].corr(df["sales"]), 3))


## 14. Correlation Analysis

In [ ]:
numeric_cols = ["quantity","sales","discount","profit","shipping_days","profit_margin"]
corr = df[numeric_cols].corr()

display(corr.round(2))

plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Retail Sales Correlation Heatmap")
plt.tight_layout()
plt.savefig(VISUAL_DIR/"12_correlation_heatmap.png", dpi=150)
plt.show()


## 15. Top & Bottom Performers

In [ ]:
top_profit_subcats = subcategory.sort_values("profit", ascending=False).head(5)
bottom_profit_subcats = subcategory.sort_values("profit", ascending=True).head(5)

print("Top 5 sub-categories by profit")
display(top_profit_subcats.round(2))

print("Bottom 5 sub-categories by profit")
display(bottom_profit_subcats.round(2))

high_discount = df[df["discount"] >= 0.30]
print(f"Orders with discount >= 30%: {len(high_discount):,}")
print(f"Average profit margin for >=30% discount: {high_discount['profit_margin'].mean():.2f}%")


## 16. Automated Business Insights

In [ ]:
best_category = category_summary["sales"].idxmax()
best_profit_category = category_summary["profit"].idxmax()
best_region = region_summary["sales"].idxmax()
best_subcat = subcategory["profit"].idxmax()
worst_subcat = subcategory["profit"].idxmin()
best_segment = segment["sales"].idxmax()
most_used_ship = shipping["orders"].idxmax()
slowest_ship = shipping["avg_shipping_days"].idxmax()

print("BUSINESS INSIGHTS")
print("="*70)
print(f"1. Highest-sales category: {best_category}")
print(f"2. Highest-profit category: {best_profit_category}")
print(f"3. Highest-sales region: {best_region}")
print(f"4. Most profitable sub-category: {best_subcat}")
print(f"5. Lowest-profit sub-category: {worst_subcat}")
print(f"6. Largest customer segment by sales: {best_segment}")
print(f"7. Most-used shipping mode: {most_used_ship}")
print(f"8. Slowest average shipping mode: {slowest_ship}")

if len(high_discount) > 0:
    print(f"9. High-discount orders (>=30%): {len(high_discount):,}")
    print(f"   Their average profit margin: {high_discount['profit_margin'].mean():.2f}%")

print("\nRecommended actions:")
print("- Review heavily discounted transactions for margin leakage.")
print("- Prioritize high-profit categories and sub-categories for growth.")
print("- Investigate consistently low-profit sub-categories.")
print("- Use monthly seasonality for inventory and staffing planning.")
print("- Monitor shipping performance alongside customer demand.")


## 17. Export Clean Dataset and Final Outputs

In [ ]:
# Save cleaned dataset
df.to_csv(OUTPUT_DIR/"cleaned_retail_sales.csv", index=False)

# Additional summaries
region_summary.to_csv(OUTPUT_DIR/"region_summary.csv")
category_summary.to_csv(OUTPUT_DIR/"category_summary.csv")
monthly.to_csv(OUTPUT_DIR/"monthly_sales_summary.csv", index=False)

print("Export completed.")
print("Files saved in:", OUTPUT_DIR.resolve())
print("Charts saved in:", VISUAL_DIR.resolve())


# Final Conclusion

This project demonstrates a complete retail analytics workflow from raw transaction data to business recommendations.

### Skills demonstrated
- Data cleaning and validation
- Pandas / NumPy
- Exploratory Data Analysis
- KPI development
- Profitability analysis
- Time-series trend analysis
- Customer segmentation
- Regional analysis
- Discount analysis
- Shipping analytics
- Data visualization
- Automated business insights

### Portfolio note
For the final GitHub version, use your actual retail dataset and re-run the notebook so the exported CSV files and charts reflect the real data.
